# Análise Exploratória do Mercado de Cinema Brasileiro

Análise exploratória dos dados de bilheteria do mercado cinematográfico nacional,
utilizando dados públicos da **ANCINE (Agência Nacional do Cinema)**.

**Período coberto:** janeiro de 2014 a junho de 2026

> ⚠️ Os dados de **2026 são parciais** — cobrem apenas de janeiro a junho.
> Todas as comparações anuais que incluam 2026 indicarão isso explicitamente.

---

## Perguntas investigadas

1. **Impacto da pandemia de COVID-19:** Como e quando o mercado foi afetado?
   Qual a magnitude da queda?
2. **Recuperação do mercado:** O público retornou aos patamares pré-pandemia?
   Em que ritmo?
3. **Filmes premiados:** Qual o impacto de *Ainda Estou Aqui* e *O Agente Secreto*
   na bilheteria do cinema nacional?
4. **Cinema nacional vs. estrangeiro:** Como evolui a participação do cinema
   brasileiro ao longo dos anos?
5. **Sazonalidade:** Existem padrões sazonais consistentes no consumo de cinema?
6. **Distribuição geográfica:** Como o público se distribui entre estados e regiões?

---

> **Sobre os dados:** Os registros da ANCINE contabilizam **público** (número de
> espectadores por sessão/dia/sala), não receita financeira. A identificação de filmes
> nacionais é feita pelo prefixo `B` no campo `CPB_ROE` (código de registro ANCINE).

---
## Seção 0 — Setup e Configurações

Nesta seção preparamos o ambiente de análise:
- Importação das bibliotecas necessárias
- Definição dos caminhos de entrada e saída
- Configurações visuais do matplotlib
- Funções auxiliares para salvar figuras e formatar eixos

In [1]:
# ── Manipulação e análise de dados ──────────────────────────
import pandas as pd
import numpy as np

# ── Visualização ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Utilitários de sistema ───────────────────────────────────
import glob
import warnings
from pathlib import Path

# Suprimir avisos não críticos para manter o output do notebook limpo
warnings.filterwarnings('ignore')

# Confirmar versões das principais bibliotecas
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')
print(f'seaborn {sns.__version__}')

pandas  2.3.3
numpy   2.0.2
seaborn 0.13.2


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CAMINHOS
# ═══════════════════════════════════════════════════════════════
# O notebook fica em notebooks/, então o diretório raiz está
# um nível acima. Path.resolve() garante o caminho absoluto.
ROOT_DIR = Path('..').resolve()

DATA_DIR        = ROOT_DIR / 'data'                 # CSVs brutos ANCINE (não versionados)
FIGURES_DIR     = ROOT_DIR / 'outputs' / 'figures'  # Gráficos exportados (versionados)
OUTPUT_DATA_DIR = ROOT_DIR / 'outputs' / 'processados'  # Dados processados/agregados (versionados)

# Criar os diretórios de saída caso não existam
# (útil quando o notebook é executado em um ambiente limpo)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dados brutos:   {DATA_DIR}')
print(f'Figuras:        {FIGURES_DIR}')
print(f'Dados gerados:  {OUTPUT_DATA_DIR}')

# ═══════════════════════════════════════════════════════════════
# CONFIGURAÇÕES DE VISUALIZAÇÃO
# ═══════════════════════════════════════════════════════════════
# Estilo base: grade de fundo clara, sem bordas superiores/direitas
plt.style.use('seaborn-v0_8-whitegrid')

# Parâmetros globais aplicados a todos os gráficos do notebook
plt.rcParams.update({
    'figure.figsize':    (14, 6),  # tamanho padrão: largo e não muito alto
    'figure.dpi':        100,
    'font.size':         12,
    'axes.titlesize':    14,
    'axes.labelsize':    12,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   11,
    'axes.spines.top':   False,  # remover bordas superior e direita
    'axes.spines.right': False,  # deixa o gráfico mais limpo visualmente
})

# Paleta de cores padronizada para o projeto.
# Usar sempre as mesmas cores para os mesmos conceitos facilita
# a leitura e a comparação entre gráficos.
CORES = {
    'nacional':    '#1f77b4',  # azul    — cinema nacional
    'estrangeiro': '#ff7f0e',  # laranja — cinema estrangeiro
    'pandemia':    '#d62728',  # vermelho — período pandemia
    'destaque':    '#2ca02c',  # verde   — filmes premiados
    'neutro':      '#7f7f7f',  # cinza   — 2026 (ano incompleto)
    'total':       '#17becf',  # ciano   — totais gerais
}

# ═══════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ═══════════════════════════════════════════════════════════════
def salvar_figura(nome: str) -> None:
    """Salva a figura matplotlib atual em outputs/figures/."""
    caminho = FIGURES_DIR / nome
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    print(f'Figura salva: outputs/figures/{nome}')


def formatar_milhoes(x, pos=None) -> str:
    """Formata valores do eixo Y em milhões (ex: 5,2M) para legibilidade."""
    return f'{x/1_000_000:.1f}M'


print('Configurações aplicadas com sucesso.')

---
## Seção 1 — Carregamento e Otimização de Memória

Os dados da ANCINE estão distribuídos em **150 arquivos CSV** mensais
(jan/2014 a jun/2026), totalizando aproximadamente **4,5 GB em disco** e
**20,9 milhões de linhas**.

### Estratégia de carregamento

Carregar tudo de uma vez exige cuidado com o uso de memória. As otimizações adotadas são:

| Técnica | Por quê | Ganho estimado |
|---------|---------|----------------|
| `dtype='category'` para colunas com poucos valores únicos | Armazena como inteiro com tabela de mapeamento | Até 90% vs. `object` |
| `dtype='int32'` para `PUBLICO` | Valores até ~2.100 — não precisa de int64 | 50% vs. `int64` |
| `dtype='Int32'` (nullable) para IDs com nulos | Evita conversão automática para `float64` | 50% vs. `float64` |
| Descartar 3 colunas não utilizadas na leitura | Reduz a quantidade de dados carregados | ~15% |
| Converter `DATA_EXIBICAO` após o concat | Parsear datas uma vez só, no DataFrame completo | Mais rápido |

**Resultado esperado:** ~1,5–2,0 GB em RAM (vs. ~4,5 GB sem otimização de tipos).

In [3]:
# ═══════════════════════════════════════════════════════════════
# TIPOS DE DADOS OTIMIZADOS
# ═══════════════════════════════════════════════════════════════
# Definir os dtypes NA LEITURA (e não depois) é essencial:
# se carregarmos como object/int64 e convertermos depois, o pandas
# mantém as duas versões na memória simultaneamente durante a conversão.

DTYPES = {
    # ── Texto com alta repetição → 'category' ─────────────────
    # 'category' armazena os dados como inteiros internamente,
    # mantendo apenas uma tabela de mapeamento para os valores únicos.
    # Ideal quando os mesmos valores se repetem milhões de vezes.

    'TITULO_ORIGINAL':            'category',  # mesmo filme em N salas × N dias
    'TITULO_BRASIL':              'category',  # mesma lógica
    'CPB_ROE':                    'category',  # código ANCINE — B=nacional, E=estrangeiro
    'PAIS_OBRA':                  'category',  # ~50 países únicos em todo o dataset
    'NOME_SALA':                  'category',  # cada sala repete todos os dias que exibe
    'MUNICIPIO_SALA_COMPLEXO':    'category',  # ~500 municípios únicos
    'UF_SALA_COMPLEXO':           'category',  # exatamente 27 valores (siglas dos estados)
    'RAZAO_SOCIAL_DISTRIBUIDORA': 'category',  # poucas dezenas de distribuidoras

    # ── Público: int32 é suficiente ───────────────────────────
    # Valores típicos: 1–1.600 espectadores por sala por dia.
    # int32 suporta até ~2,1 bilhões — mais que suficiente.
    'PUBLICO': 'int32',

    # ── IDs numéricos com nulos ───────────────────────────────
    # Int32 (maiúsculo) é a versão nullable do pandas (>=1.0).
    # Sem ele, o pandas converteria automaticamente para float64
    # na presença de qualquer nulo, dobrando o consumo de memória.
    'REGISTRO_SALA':           'Int32',  # 0,16% de nulos em anos antigos
    'REGISTRO_COMPLEXO':       'Int32',
    'REGISTRO_EXIBIDOR':       'Int32',
    'REGISTRO_GRUPO_EXIBIDOR': 'Int32',  # 8,6% de nulos
}

# Colunas sem utilidade para a análise — descartadas durante a leitura
COLUNAS_DESCARTAR = [
    'NR_PROTOCOLO_ENVIO',         # número interno do protocolo ANCINE
    'DATA_HORA_ENVIO_PROTOCOLO',  # timestamp do envio pelo exibidor (≠ data de exibição)
    'CNPJ_DISTRIBUIDORA',         # redundante — já temos RAZAO_SOCIAL_DISTRIBUIDORA
]

print(f'Dtypes otimizados para {len(DTYPES)} colunas')
print(f'Colunas descartadas na leitura: {COLUNAS_DESCARTAR}')

Dtypes otimizados para 13 colunas
Colunas descartadas na leitura: ['NR_PROTOCOLO_ENVIO', 'DATA_HORA_ENVIO_PROTOCOLO', 'CNPJ_DISTRIBUIDORA']


In [4]:
# ═══════════════════════════════════════════════════════════════
# CARREGAMENTO DOS ARQUIVOS CSV
# ═══════════════════════════════════════════════════════════════
# Localizar todos os arquivos no diretório de dados.
# sorted() garante leitura em ordem cronológica (jan/2014 → jun/2026).
arquivos = sorted(glob.glob(str(DATA_DIR / '*.csv')))

print(f'Arquivos encontrados: {len(arquivos)}')
print(f'  Primeiro: {Path(arquivos[0]).name}')
print(f'  Último:   {Path(arquivos[-1]).name}')
print()

# ── Carregamento em lista + concat ao final ───────────────────
# Por que não concat em loop?
# pd.concat dentro de um loop é O(n²) — cada iteração cria uma
# cópia completa do DataFrame acumulado. Com 150 arquivos, isso
# seria extremamente lento e desperdiçaria muita memória.
# A abordagem correta: guardar tudo em lista e fazer um único concat.
print('Carregando arquivos... (pode levar alguns minutos)')

dfs = []
for i, caminho in enumerate(arquivos, 1):
    df_mes = pd.read_csv(
        caminho,
        sep=';',                                        # separador padrão ANCINE
        encoding='utf-8',                              # verificado em todos os 150 arquivos
        dtype=DTYPES,                                  # tipos otimizados definidos acima
        usecols=lambda c: c not in COLUNAS_DESCARTAR, # descartar na leitura, não depois
    )
    dfs.append(df_mes)

    # Exibir progresso a cada 10 arquivos e ao final
    if i % 10 == 0 or i == len(arquivos):
        print(f'  {i:>3}/{len(arquivos)} arquivos carregados...')

# Concatenar tudo em um único DataFrame
# ignore_index=True: reconstrói o índice de 0 a N (evita índices duplicados)
df = pd.concat(dfs, ignore_index=True)

# Liberar a lista intermediária para recuperar a memória alocada
del dfs

print(f'\nCarregamento concluído!')
print(f'  Linhas totais: {len(df):>12,}')
print(f'  Colunas:       {df.shape[1]:>12}')

Arquivos encontrados: 150
  Primeiro: bilheteria-diaria-obras-por-distribuidoras-2014-01.csv
  Último:   bilheteria-diaria-obras-por-distribuidoras-2026-06.csv

Carregando arquivos... (pode levar alguns minutos)
   10/150 arquivos carregados...
   20/150 arquivos carregados...
   30/150 arquivos carregados...
   40/150 arquivos carregados...
   50/150 arquivos carregados...
   60/150 arquivos carregados...
   70/150 arquivos carregados...
   80/150 arquivos carregados...
   90/150 arquivos carregados...
  100/150 arquivos carregados...
  110/150 arquivos carregados...
  120/150 arquivos carregados...
  130/150 arquivos carregados...
  140/150 arquivos carregados...
  150/150 arquivos carregados...

Carregamento concluído!
  Linhas totais:   20,916,062
  Colunas:                 14


In [5]:
# ═══════════════════════════════════════════════════════════════
# CONVERSÃO DA COLUNA DE DATA
# ═══════════════════════════════════════════════════════════════
# DATA_EXIBICAO foi lida como string no formato DD/MM/AAAA.
# Convertemos para datetime APÓS o concat porque é mais eficiente:
# parsear datas uma vez no DataFrame completo é mais rápido do que
# parsear em cada um dos 150 arquivos individualmente.
#
# O formato explícito '%d/%m/%Y' é mais rápido do que inferência automática.
df['DATA_EXIBICAO'] = pd.to_datetime(df['DATA_EXIBICAO'], format='%d/%m/%Y')

# Confirmar o período coberto
data_inicio = df['DATA_EXIBICAO'].min().date()
data_fim    = df['DATA_EXIBICAO'].max().date()
print(f'Período coberto: {data_inicio}  →  {data_fim}')
print()

# ═══════════════════════════════════════════════════════════════
# DIAGNÓSTICO DE MEMÓRIA
# ═══════════════════════════════════════════════════════════════
# memory_usage(deep=True) considera o conteúdo real das colunas
# de tipo object/category (não apenas os ponteiros), dando um
# número preciso do consumo real em memória.
mem_atual_mb = df.memory_usage(deep=True).sum() / 1_048_576

# Tamanho total em disco como referência de comparação
tamanho_disco_mb = sum(
    Path(f).stat().st_size for f in arquivos
) / 1_048_576

print('Uso de memória do DataFrame:')
print(f'  Em memória (otimizado): {mem_atual_mb:>8.0f} MB')
print(f'  Em disco (referência):  {tamanho_disco_mb:>8.0f} MB')
print()

# Detalhe por coluna — identifica as que mais consomem memória
mem_por_coluna = (
    df.memory_usage(deep=True)
      .drop('Index')              # excluir o índice do ranking
      .sort_values(ascending=False)
      .head(8)                    # top 8 colunas mais pesadas
      .apply(lambda x: f'{x/1_048_576:.1f} MB')
)
print('Top 8 colunas por consumo de memória:')
print(mem_por_coluna.to_string())

Período coberto: 2014-01-01  →  2026-06-13

Uso de memória do DataFrame:
  Em memória (otimizado):    13300 MB
  Em disco (referência):      4637 MB

Top 8 colunas por consumo de memória:
RAZAO_SOCIAL_DISTRIBUIDORA    1873.8 MB
NOME_SALA                     1862.6 MB
TITULO_BRASIL                 1741.2 MB
MUNICIPIO_SALA_COMPLEXO       1623.2 MB
TITULO_ORIGINAL               1564.8 MB
PAIS_OBRA                     1416.8 MB
CPB_ROE                       1416.2 MB
UF_SALA_COMPLEXO              1163.4 MB


---
## Seção 2 — Limpeza e Padronização dos Dados

Antes de qualquer análise, aplicamos três grupos de transformações:

1. **Remoção de registros inválidos** — público zero presente apenas em 2014
2. **Preenchimento de valores ausentes** — `TITULO_BRASIL` vazio em 4%–31% das linhas
3. **Correção de acentuação em `PAIS_OBRA`** — único problema de padronização
   encontrado nos dados (ex: `JAPAO` vs `JAPÃO`)
4. **Criação de colunas derivadas** — facilitam os agrupamentos temporais e de origem

> **Sobre a qualidade geral dos dados:** exceto pela acentuação inconsistente de `PAIS_OBRA`,
> os dados ANCINE são bem padronizados — mesmas siglas de UF, municípios com acentuação
> consistente, e títulos de filmes sem variações de grafia.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2.1 — REMOÇÃO DE REGISTROS COM PÚBLICO ZERO
# ═══════════════════════════════════════════════════════════════
# Em 2014, cerca de 7.400 linhas (~5,7% daquele ano) têm PUBLICO == 0.
# A partir de 2015, esse problema desaparece completamente.
# Esses registros parecem ser testes ou exibições sem público registrado
# e seriam ruído nas análises de público.

registros_antes = len(df)
zeros_por_ano = df[df['PUBLICO'] == 0].groupby(df['DATA_EXIBICAO'].dt.year).size()

if len(zeros_por_ano) > 0:
    print('Registros com público zero por ano:')
    print(zeros_por_ano.to_string())
    print()

# Remover todas as linhas com público zero
df = df[df['PUBLICO'] > 0].copy()

registros_removidos = registros_antes - len(df)
print(f'Registros removidos (público zero): {registros_removidos:,}')
print(f'Registros restantes:                {len(df):,}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2.2 — PREENCHIMENTO DE TITULO_BRASIL
# ═══════════════════════════════════════════════════════════════
# TITULO_BRASIL está vazio em 4% a 31% das linhas dependendo do ano.
# O percentual varia porque:
#   - Filmes nacionais frequentemente não têm título traduzido
#   - Filmes mais antigos têm mais campos faltantes nos registros ANCINE
# Solução: usar TITULO_ORIGINAL quando TITULO_BRASIL está ausente.

# Percentual de nulos por ano antes da correção
nulos_antes = df.groupby(df['DATA_EXIBICAO'].dt.year)['TITULO_BRASIL'].apply(
    lambda x: x.isna().mean() * 100
).round(1)
print('% de TITULO_BRASIL vazio por ano (antes):')
print(nulos_antes.to_string())
print()

# Para colunas do tipo 'category', precisamos adicionar os valores do
# TITULO_ORIGINAL às categorias de TITULO_BRASIL antes de preencher,
# caso contrário o pandas levanta erro de categoria inexistente.
df['TITULO_BRASIL'] = df['TITULO_BRASIL'].cat.add_categories(
    [c for c in df['TITULO_ORIGINAL'].cat.categories
     if c not in df['TITULO_BRASIL'].cat.categories]
)
df['TITULO_BRASIL'] = df['TITULO_BRASIL'].fillna(df['TITULO_ORIGINAL'])

nulos_depois = df['TITULO_BRASIL'].isna().sum()
print(f'Nulos em TITULO_BRASIL após preenchimento: {nulos_depois}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2.3 — CORREÇÃO DE ACENTUAÇÃO EM PAIS_OBRA
# ═══════════════════════════════════════════════════════════════
# Problema encontrado na exploração: alguns países perdem acentos
# em certos arquivos mensais — provavelmente por diferenças no
# processo de geração dos CSVs ao longo dos anos.
# Exemplos encontrados: 'JAPAO' (deveria ser 'JAPÃO'),
#                       'FRANCA' (deveria ser 'FRANÇA'),
#                       'ITALIA' (deveria ser 'ITÁLIA')

# Mapeamento de correção: forma incorreta → forma correta
CORRECOES_PAIS = {
    'JAPAO':  'JAPÃO',
    'FRANCA': 'FRANÇA',
    'ITALIA': 'ITÁLIA',
    'LIBANO': 'LÍBANO',
}

# Verificar quais formas incorretas realmente existem no dataset
paises_unicos = df['PAIS_OBRA'].cat.categories.tolist()
correcoes_aplicaveis = {k: v for k, v in CORRECOES_PAIS.items() if k in paises_unicos}

if correcoes_aplicaveis:
    print('Correções a aplicar:')
    for errado, correto in correcoes_aplicaveis.items():
        qtd = (df['PAIS_OBRA'] == errado).sum()
        print(f'  {errado!r:12s} → {correto!r:12s}  ({qtd:,} registros)')

    # Para category, renameamos as categorias diretamente
    # (mais eficiente do que usar replace linha a linha)
    df['PAIS_OBRA'] = df['PAIS_OBRA'].cat.rename_categories(
        {k: v for k, v in correcoes_aplicaveis.items()}
    )
    print(f'\nCorreções aplicadas.')
else:
    print('Nenhuma forma incorreta encontrada — dados já estão corretos.')

print(f'\nTotal de países únicos após correção: {df["PAIS_OBRA"].nunique()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2.4 — COLUNAS DERIVADAS
# ═══════════════════════════════════════════════════════════════
# Criamos colunas calculadas que facilitam os agrupamentos
# sem precisar recalcular em cada análise.

# ORIGEM: identifica se o filme é nacional ou estrangeiro.
# O campo CPB_ROE começa com 'B' para filmes brasileiros
# e com 'E' para estrangeiros (padrão de registro ANCINE).
df['ORIGEM'] = df['CPB_ROE'].str[0].map(
    {'B': 'Nacional', 'E': 'Estrangeiro'}
).astype('category')

# ANO e MES: extraídos de DATA_EXIBICAO para facilitar groupbys
df['ANO'] = df['DATA_EXIBICAO'].dt.year.astype('int16')   # int16: 2014–2026 cabe fácil
df['MES'] = df['DATA_EXIBICAO'].dt.month.astype('int8')   # int8: 1–12

# ANO_MES: período mensal — ideal para séries temporais
# Period é mais eficiente do que string 'YYYY-MM' para groupbys
df['ANO_MES'] = df['DATA_EXIBICAO'].dt.to_period('M')

# ANO_COMPLETO: flag para separar 2026 (incompleto) nas análises anuais.
# 2026 tem dados apenas até junho — incluí-lo em totais anuais
# distorceria comparações com anos anteriores completos.
df['ANO_COMPLETO'] = (df['ANO'] < 2026)

print('Colunas derivadas criadas:')
print(f'  ORIGEM:         {df["ORIGEM"].value_counts().to_dict()}')
print(f'  ANO:            {df["ANO"].min()} → {df["ANO"].max()}')
print(f'  MES:            {df["MES"].min()} → {df["MES"].max()}')
print(f'  ANO_COMPLETO:   {df["ANO_COMPLETO"].value_counts().to_dict()}')
print(f'\nShape final do DataFrame: {df.shape}')

---
## Seção 3 — Visão Geral do Dataset

Com os dados limpos e padronizados, exploramos as dimensões gerais do dataset:
quantos filmes, salas, municípios e estados cobertos; e como o volume de registros
se distribui ao longo dos anos — o que já revela visualmente o impacto da pandemia
e o dado incompleto de 2026.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.1 — MÉTRICAS GERAIS DO DATASET
# ═══════════════════════════════════════════════════════════════
print('=' * 50)
print('RESUMO DO DATASET ANCINE')
print('=' * 50)
print(f'Período:                {df["DATA_EXIBICAO"].min().date()}  →  {df["DATA_EXIBICAO"].max().date()}')
print(f'Total de registros:     {len(df):>12,}  (linha = 1 sala × 1 dia × 1 filme)')
print(f'Público total:          {df["PUBLICO"].sum():>12,}  espectadores')
print()
print(f'Filmes únicos:          {df["TITULO_ORIGINAL"].nunique():>12,}')
print(f'  Nacional:             {df.loc[df["ORIGEM"]=="Nacional", "TITULO_ORIGINAL"].nunique():>12,}')
print(f'  Estrangeiro:          {df.loc[df["ORIGEM"]=="Estrangeiro", "TITULO_ORIGINAL"].nunique():>12,}')
print()
print(f'Salas únicas:           {df["NOME_SALA"].nunique():>12,}')
print(f'Municípios únicos:      {df["MUNICIPIO_SALA_COMPLEXO"].nunique():>12,}')
print(f'Estados (UFs):          {df["UF_SALA_COMPLEXO"].nunique():>12,}')
print(f'Distribuidoras:         {df["RAZAO_SOCIAL_DISTRIBUIDORA"].nunique():>12,}')
print(f'Países de origem:       {df["PAIS_OBRA"].nunique():>12,}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.2 — PÚBLICO E REGISTROS POR ANO
# ═══════════════════════════════════════════════════════════════
# Esta tabela já deixa claro o impacto da pandemia (2020-2021)
# e o dado incompleto de 2026 (apenas jan-jun).

resumo_anual = df.groupby('ANO', observed=True).agg(
    Registros=('PUBLICO', 'count'),
    Publico_Total=('PUBLICO', 'sum'),
    Publico_Medio_por_Sessao=('PUBLICO', 'mean'),
).reset_index()

# Calcular variação percentual de público em relação ao ano anterior
resumo_anual['Var_Pct_YoY'] = resumo_anual['Publico_Total'].pct_change() * 100

# Formatar para exibição
resumo_display = resumo_anual.copy()
resumo_display['Publico_Total']            = resumo_display['Publico_Total'].apply(lambda x: f'{x:,.0f}')
resumo_display['Registros']                = resumo_display['Registros'].apply(lambda x: f'{x:,.0f}')
resumo_display['Publico_Medio_por_Sessao'] = resumo_display['Publico_Medio_por_Sessao'].apply(lambda x: f'{x:.1f}')
resumo_display['Var_Pct_YoY']              = resumo_display['Var_Pct_YoY'].apply(
    lambda x: f'{x:+.1f}%' if not (x != x) else '—'  # NaN no primeiro ano
)

# Adicionar nota para 2026 (ano incompleto)
resumo_display.loc[resumo_display['ANO'] == 2026, 'ANO'] = '2026 *'

resumo_display.columns = ['Ano', 'Registros', 'Público Total', 'Média por Sessão', 'Var. YoY']
print(resumo_display.to_string(index=False))
print()
print('* 2026: dados de janeiro a junho apenas')

# Salvar para reutilização no dashboard
resumo_anual.to_csv(OUTPUT_DATA_DIR / 'publico_anual.csv', index=False)
print('\nTabela salva em outputs/processados/publico_anual.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.3 — DISTRIBUIÇÃO POR ORIGEM: NACIONAL vs. ESTRANGEIRO
# ═══════════════════════════════════════════════════════════════
# Visão geral da proporção entre cinema nacional e estrangeiro
# ao longo de todo o período.

por_origem = df.groupby('ORIGEM', observed=True)['PUBLICO'].agg(
    Total='sum', Registros='count'
).reset_index()

por_origem['Pct_Publico'] = por_origem['Total'] / por_origem['Total'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico 1: Distribuição de público total
axes[0].pie(
    por_origem['Total'],
    labels=por_origem['ORIGEM'],
    colors=[CORES['nacional'], CORES['estrangeiro']],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12},
)
axes[0].set_title('Distribuição do Público Total (2014–2026)', fontweight='bold')

# Gráfico 2: Número de filmes únicos por origem
filmes_por_origem = df.groupby('ORIGEM', observed=True)['TITULO_ORIGINAL'].nunique()
bars = axes[1].bar(
    filmes_por_origem.index,
    filmes_por_origem.values,
    color=[CORES['nacional'], CORES['estrangeiro']],
    edgecolor='white',
    linewidth=1.5,
    width=0.5,
)
# Adicionar rótulos de valor sobre as barras
for bar, val in zip(bars, filmes_por_origem.values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f'{val:,}',
        ha='center', va='bottom', fontweight='bold', fontsize=11,
    )
axes[1].set_title('Número de Títulos Únicos por Origem', fontweight='bold')
axes[1].set_ylabel('Quantidade de filmes')
axes[1].set_xlabel('')

plt.suptitle('Visão Geral: Cinema Nacional vs. Estrangeiro (2014–2026)', fontsize=14, y=1.02)
plt.tight_layout()
salvar_figura('00_visao_geral_origem.png')
plt.show()

---
## Seção 4 — Evolução Temporal do Mercado (2014–2026)

Visão macro da evolução do público de cinema no Brasil ao longo de mais de uma
década. Os gráficos mostram tanto a série mensal completa quanto os totais anuais,
já evidenciando o colapso de 2020 e a trajetória de recuperação posterior.

> **Atenção — 2026 incompleto:** o ano de 2026 tem dados apenas de janeiro a junho.
> Nos gráficos anuais, 2026 aparece em cinza com a nota `(jan–jun)` para deixar
> claro que não é comparável aos anos anteriores.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.1 — SÉRIE TEMPORAL MENSAL (2014–2026)
# ═══════════════════════════════════════════════════════════════
# Agregar público total por mês
publico_mensal = (
    df.groupby('ANO_MES', observed=True)['PUBLICO']
      .sum()
      .reset_index()
)
# Converter Period para timestamp para compatibilidade com matplotlib
publico_mensal['DATA'] = publico_mensal['ANO_MES'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    publico_mensal['DATA'],
    publico_mensal['PUBLICO'],
    color=CORES['total'],
    linewidth=1.5,
    label='Público mensal',
)

# Destacar o período de pandemia com fundo vermelho translúcido
# Cinemas fecharam em março/2020 e reabriram gradualmente até 2022
import pandas as pd
ax.axvspan(
    pd.Timestamp('2020-03-01'),
    pd.Timestamp('2021-12-31'),
    alpha=0.12, color=CORES['pandemia'], label='Período pandemia',
)

# Anotações dos eventos mais importantes
ax.annotate(
    'Fechamento\ndas salas\n(mar/2020)',
    xy=(pd.Timestamp('2020-04-01'), 50_000),
    xytext=(pd.Timestamp('2019-04-01'), 8_000_000),
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
    fontsize=9, color='gray', ha='center',
)
ax.annotate(
    'Ainda Estou Aqui\n(nov/2024)',
    xy=(pd.Timestamp('2024-11-01'), publico_mensal.loc[
        publico_mensal['ANO_MES'] == pd.Period('2024-11', 'M'), 'PUBLICO'
    ].values[0]),
    xytext=(pd.Timestamp('2023-06-01'), 22_000_000),
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
    fontsize=9, color=CORES['destaque'], ha='center',
)

# Eixo Y em milhões
ax.yaxis.set_major_formatter(mticker.FuncFormatter(formatar_milhoes))
ax.set_title('Público Mensal de Cinema no Brasil (2014–2026)', fontweight='bold', fontsize=15)
ax.set_xlabel('')
ax.set_ylabel('Público (milhões de espectadores)')
ax.legend(loc='upper left')

plt.tight_layout()
salvar_figura('01_publico_mensal_historico.png')
plt.show()

# Salvar série mensal
publico_mensal[['ANO_MES', 'PUBLICO']].to_csv(
    OUTPUT_DATA_DIR / 'publico_mensal.csv', index=False
)
print('Série mensal salva.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.2 — PÚBLICO ANUAL (BARRAS)
# ═══════════════════════════════════════════════════════════════
# Agregar por ano. 2026 fica separado visualmente por ser incompleto.
publico_anual = df.groupby('ANO', observed=True)['PUBLICO'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))

# Cor diferente para 2026 (ano incompleto) e para 2020-2021 (pandemia)
cores_barras = []
for ano in publico_anual['ANO']:
    if ano == 2026:
        cores_barras.append(CORES['neutro'])
    elif ano in (2020, 2021):
        cores_barras.append(CORES['pandemia'])
    else:
        cores_barras.append(CORES['total'])

bars = ax.bar(
    publico_anual['ANO'].astype(str),
    publico_anual['PUBLICO'],
    color=cores_barras,
    edgecolor='white',
    linewidth=0.8,
)

# Rótulo de valor sobre cada barra (em milhões)
for bar, val in zip(bars, publico_anual['PUBLICO']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500_000,
        f'{val/1_000_000:.0f}M',
        ha='center', va='bottom', fontsize=8.5, fontweight='bold',
    )

# Rótulo especial para 2026 indicando que é parcial
idx_2026 = publico_anual[publico_anual['ANO'] == 2026].index
if len(idx_2026) > 0:
    bar_2026 = bars[idx_2026[0]]
    ax.text(
        bar_2026.get_x() + bar_2026.get_width() / 2,
        -8_000_000,
        '(jan–jun)',
        ha='center', va='top', fontsize=8, color=CORES['neutro'], style='italic',
    )

# Legenda manual das cores
from matplotlib.patches import Patch
legenda = [
    Patch(color=CORES['total'],   label='Anos completos'),
    Patch(color=CORES['pandemia'], label='Pandemia (2020–2021)'),
    Patch(color=CORES['neutro'],   label='2026 (jan–jun, incompleto)'),
]
ax.legend(handles=legenda, loc='upper left')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(formatar_milhoes))
ax.set_title('Público Anual de Cinema no Brasil (2014–2026)', fontweight='bold', fontsize=15)
ax.set_xlabel('Ano')
ax.set_ylabel('Público total (milhões)')

plt.tight_layout()
salvar_figura('02_publico_anual.png')
plt.show()

---
## Seção 5 — Impacto da Pandemia de COVID-19

A pandemia de COVID-19 causou o colapso mais abrupto já registrado no mercado
cinematográfico brasileiro. Em março de 2020, os cinemas foram fechados por
determinação do governo. Entre abril e agosto de 2020, praticamente não houve
exibições — os poucos registros correspondem a Estados que reabriram antes.

Esta seção analisa a magnitude da queda e a velocidade de recuperação mensal,
comparando o período 2019–2022 contra o baseline pré-pandemia.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5.1 — ZOOM NO PERÍODO 2019–2022
# ═══════════════════════════════════════════════════════════════
import pandas as pd

# Filtrar apenas o período de interesse para a análise da pandemia
periodo_covid = publico_mensal[
    (publico_mensal['DATA'] >= '2019-01-01') &
    (publico_mensal['DATA'] <= '2022-12-31')
].copy()

fig, ax = plt.subplots(figsize=(16, 6))

# Linha do público mensal
ax.plot(
    periodo_covid['DATA'],
    periodo_covid['PUBLICO'],
    color=CORES['total'],
    linewidth=2,
    marker='o', markersize=4,
    label='Público mensal',
)

# Área sombreada: período de fechamento total (mar–ago/2020)
ax.axvspan(
    pd.Timestamp('2020-03-01'), pd.Timestamp('2020-08-31'),
    alpha=0.2, color=CORES['pandemia'], label='Fechamento total',
)
# Área sombreada: reabertura parcial (set/2020–dez/2021)
ax.axvspan(
    pd.Timestamp('2020-09-01'), pd.Timestamp('2021-12-31'),
    alpha=0.08, color='orange', label='Reabertura gradual',
)

# Linha de referência: média mensal de 2019 (baseline pré-pandemia)
media_2019 = publico_mensal[
    publico_mensal['ANO_MES'].dt.year == 2019
]['PUBLICO'].mean()
ax.axhline(
    media_2019, color='gray', linestyle='--', linewidth=1.2, alpha=0.7,
    label=f'Média mensal 2019 ({media_2019/1_000_000:.1f}M)',
)

# Anotações dos marcos principais
anotacoes = [
    ('2020-03-15', 50_000,   '2020-02-01', 9_500_000,  'Cinemas fecham\n(mar/2020)'),
    ('2020-09-01', 800_000,  '2020-07-01', 6_000_000,  'Reabertura\nparcial\n(set/2020)'),
    ('2022-01-01', 9_000_000,'2021-09-01', 14_000_000, 'Retomada plena\n(2022)'),
]
for xp, yp, xt, yt, texto in anotacoes:
    ax.annotate(
        texto,
        xy=(pd.Timestamp(xp), yp),
        xytext=(pd.Timestamp(xt), yt),
        arrowprops=dict(arrowstyle='->', color='gray', lw=1.2),
        fontsize=9, color='gray', ha='center',
    )

ax.yaxis.set_major_formatter(mticker.FuncFormatter(formatar_milhoes))
ax.set_title('Impacto da Pandemia de COVID-19 no Mercado de Cinema (2019–2022)',
             fontweight='bold', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Público (milhões de espectadores)')
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
salvar_figura('03_impacto_covid.png')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5.2 — MÉTRICAS QUANTITATIVAS DA QUEDA
# ═══════════════════════════════════════════════════════════════
# Comparar os totais anuais contra 2019 (baseline pré-pandemia)

pub_anual = df.groupby('ANO', observed=True)['PUBLICO'].sum()
baseline_2019 = pub_anual[2019]

print('Impacto da pandemia — público anual vs. 2019 (baseline):')
print(f'  2019 (baseline):  {baseline_2019:>12,.0f}')
print()
for ano in [2020, 2021, 2022]:
    pub = pub_anual[ano]
    variacao = (pub - baseline_2019) / baseline_2019 * 100
    print(f'  {ano}: {pub:>12,.0f}  ({variacao:+.1f}% vs 2019)')

# Mês com menor público em toda a série histórica
mes_min = publico_mensal.loc[publico_mensal['PUBLICO'].idxmin()]
print()
print(f'Mês com menor público na série histórica:')
print(f'  {mes_min["ANO_MES"]}  →  {mes_min["PUBLICO"]:,.0f} espectadores')

# Salvar tabela de métricas da queda
queda = pd.DataFrame({
    'ANO': [2019, 2020, 2021, 2022],
    'PUBLICO': [pub_anual[a] for a in [2019, 2020, 2021, 2022]],
})
queda['VAR_PCT_VS_2019'] = (queda['PUBLICO'] - baseline_2019) / baseline_2019 * 100
queda.to_csv(OUTPUT_DATA_DIR / 'queda_pandemia.csv', index=False)
print('\nTabela salva em outputs/processados/queda_pandemia.csv')

---
## Seção 6 — Recuperação do Mercado Pós-Pandemia

Com os cinemas reabrindo gradualmente a partir de 2022, o mercado entrou em
trajetória de recuperação. Mas recuperar o público pré-pandemia levou tempo:
a combinação de hábitos alterados (streaming), crise econômica e ausência de
grandes lançamentos nos primeiros anos pós-pandemia fez a recuperação ser lenta.

> **Nota metodológica:** 2026 é excluído desta análise por ser incompleto
> (apenas jan–jun). Incluí-lo distorceria a comparação com anos completos.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6.1 — TRAJETÓRIA DE RECUPERAÇÃO (2019–2025)
# ═══════════════════════════════════════════════════════════════
# Usar apenas anos completos para comparação justa
anos_completos = df[df['ANO_COMPLETO']].groupby('ANO', observed=True)['PUBLICO'].sum()

# Calcular % de recuperação em relação ao baseline 2019
baseline = anos_completos[2019]
pct_recuperacao = (anos_completos / baseline * 100).reset_index()
pct_recuperacao.columns = ['ANO', 'PCT']

fig, ax = plt.subplots(figsize=(14, 6))

# Colorir os pontos: abaixo de 100% = vermelho, acima = verde
cores_pontos = [
    CORES['pandemia'] if pct < 100 else CORES['destaque']
    for pct in pct_recuperacao['PCT']
]

ax.plot(
    pct_recuperacao['ANO'].astype(str),
    pct_recuperacao['PCT'],
    color='steelblue', linewidth=2, zorder=2,
)
ax.scatter(
    pct_recuperacao['ANO'].astype(str),
    pct_recuperacao['PCT'],
    color=cores_pontos, s=80, zorder=3,
)

# Linha de referência em 100% (nível pré-pandemia)
ax.axhline(100, color='gray', linestyle='--', linewidth=1.5, alpha=0.8,
           label='Nível pré-pandemia (2019 = 100%)')

# Rótulo com o percentual em cada ponto
for _, row in pct_recuperacao.iterrows():
    ax.text(
        str(int(row['ANO'])),
        row['PCT'] + 2.5,
        f'{row["PCT"]:.0f}%',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
        color=CORES['pandemia'] if row['PCT'] < 100 else CORES['destaque'],
    )

ax.set_ylim(0, 130)
ax.set_title('Recuperação do Público de Cinema Pós-Pandemia\n(% em relação a 2019)',
             fontweight='bold', fontsize=14)
ax.set_xlabel('Ano')
ax.set_ylabel('% do público de 2019')
ax.legend()

plt.tight_layout()
salvar_figura('04_recuperacao_pos_pandemia.png')
plt.show()

# Imprimir resumo
print('Recuperação em relação a 2019:')
for _, row in pct_recuperacao.iterrows():
    status = '✓ recuperado' if row['PCT'] >= 100 else f'{100 - row["PCT"]:.0f}pp abaixo'
    print(f'  {int(row["ANO"])}: {row["PCT"]:6.1f}%  — {status}')

---
## Seção 7 — Cinema Nacional vs. Estrangeiro

O mercado cinematográfico brasileiro é historicamente dominado por produções
estrangeiras — em especial americanas, que respondem por mais de 90% do público
estrangeiro. A participação do cinema nacional oscila bastante de ano para ano,
sendo fortemente influenciada por poucos filmes de grande bilheteria.

Esta seção analisa:
1. A evolução do **market share** do cinema nacional ao longo do tempo
2. Os **top 10 filmes** mais assistidos em cada categoria
3. A **dominância americana** no segmento estrangeiro

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.1 — MARKET SHARE ANUAL: NACIONAL vs. ESTRANGEIRO
# ═══════════════════════════════════════════════════════════════
# Agregar público por ano e origem
share_anual = (
    df.groupby(['ANO', 'ORIGEM'], observed=True)['PUBLICO']
      .sum()
      .unstack('ORIGEM')
      .fillna(0)
)

# Calcular percentual de cada origem no total do ano
share_pct = share_anual.div(share_anual.sum(axis=1), axis=0) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# ── Gráfico superior: barras empilhadas (público absoluto) ────
anos = share_anual.index.astype(str)
ax1.bar(anos, share_anual['Nacional'],    color=CORES['nacional'],    label='Nacional')
ax1.bar(anos, share_anual['Estrangeiro'], color=CORES['estrangeiro'], label='Estrangeiro',
        bottom=share_anual['Nacional'])

# Destacar 2026 como parcial
ax1.axvline(x=len(anos) - 1, color=CORES['neutro'], linestyle=':', linewidth=1.5, alpha=0.7)
ax1.text(len(anos) - 1, share_anual.sum(axis=1).max() * 1.02,
         'jan–jun', ha='center', fontsize=8, color=CORES['neutro'], style='italic')

ax1.yaxis.set_major_formatter(mticker.FuncFormatter(formatar_milhoes))
ax1.set_title('Público por Ano e Origem (absoluto)', fontweight='bold')
ax1.set_ylabel('Público total (milhões)')
ax1.legend(loc='upper left')
ax1.set_xlabel('')

# ── Gráfico inferior: barras empilhadas (percentual 100%) ─────
ax2.bar(anos, share_pct['Nacional'],    color=CORES['nacional'],    label='Nacional')
ax2.bar(anos, share_pct['Estrangeiro'], color=CORES['estrangeiro'], label='Estrangeiro',
        bottom=share_pct['Nacional'])

# Rótulo do percentual nacional em cada barra
for i, (ano, row) in enumerate(share_pct.iterrows()):
    pct_nac = row['Nacional']
    if pct_nac >= 5:  # só exibir rótulo se houver espaço
        ax2.text(i, pct_nac / 2, f'{pct_nac:.0f}%',
                 ha='center', va='center', fontsize=8,
                 color='white', fontweight='bold')

ax2.set_ylim(0, 100)
ax2.set_title('Market Share por Ano (% do público)', fontweight='bold')
ax2.set_ylabel('% do público total')
ax2.set_xlabel('Ano')
ax2.legend(loc='upper right')

plt.suptitle('Cinema Nacional vs. Estrangeiro — Evolução do Market Share',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
salvar_figura('05_market_share_nacional_estrangeiro.png')
plt.show()

# Salvar para dashboard
share_out = share_anual.copy()
share_out['pct_nacional']    = share_pct['Nacional']
share_out['pct_estrangeiro'] = share_pct['Estrangeiro']
share_out.to_csv(OUTPUT_DATA_DIR / 'market_share_anual.csv')
print('Market share salvo em outputs/processados/market_share_anual.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.2 — TOP 10 FILMES NACIONAIS (TODO O PERÍODO)
# ═══════════════════════════════════════════════════════════════
# Para identificar o título de exibição de cada filme, usamos
# TITULO_BRASIL (que foi preenchido com TITULO_ORIGINAL quando vazio).
# Agrupamos por TITULO_ORIGINAL para não contar versões do mesmo filme
# em separado caso o título em português tenha variado.

top_nacionais = (
    df[df['ORIGEM'] == 'Nacional']
      .groupby('TITULO_ORIGINAL', observed=True)['PUBLICO']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)
top_nacionais.columns = ['Título', 'Público']

fig, ax = plt.subplots(figsize=(12, 6))

# Barras horizontais — mais fáceis de ler com títulos longos
bars = ax.barh(
    range(len(top_nacionais)),
    top_nacionais['Público'],
    color=CORES['nacional'],
    edgecolor='white', linewidth=0.5,
)

# Rótulo de valor ao lado de cada barra
for bar, val in zip(bars, top_nacionais['Público']):
    ax.text(
        bar.get_width() + top_nacionais['Público'].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val/1_000_000:.1f}M',
        va='center', fontsize=9, fontweight='bold',
    )

# Títulos no eixo Y — capitalizar para leitura mais agradável
titulos_fmt = [t.title() for t in top_nacionais['Título']]
ax.set_yticks(range(len(top_nacionais)))
ax.set_yticklabels(titulos_fmt, fontsize=10)
ax.invert_yaxis()  # maior público no topo

ax.xaxis.set_major_formatter(mticker.FuncFormatter(formatar_milhoes))
ax.set_title('Top 10 Filmes Nacionais por Público (2014–2026)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Público total (milhões de espectadores)')

plt.tight_layout()
salvar_figura('06_top10_filmes_nacionais.png')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.3 — TOP 10 FILMES ESTRANGEIROS (TODO O PERÍODO)
# ═══════════════════════════════════════════════════════════════
top_estrangeiros = (
    df[df['ORIGEM'] == 'Estrangeiro']
      .groupby('TITULO_ORIGINAL', observed=True)['PUBLICO']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)
top_estrangeiros.columns = ['Título', 'Público']

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    range(len(top_estrangeiros)),
    top_estrangeiros['Público'],
    color=CORES['estrangeiro'],
    edgecolor='white', linewidth=0.5,
)
for bar, val in zip(bars, top_estrangeiros['Público']):
    ax.text(
        bar.get_width() + top_estrangeiros['Público'].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val/1_000_000:.1f}M',
        va='center', fontsize=9, fontweight='bold',
    )

titulos_fmt = [t.title() for t in top_estrangeiros['Título']]
ax.set_yticks(range(len(top_estrangeiros)))
ax.set_yticklabels(titulos_fmt, fontsize=10)
ax.invert_yaxis()

ax.xaxis.set_major_formatter(mticker.FuncFormatter(formatar_milhoes))
ax.set_title('Top 10 Filmes Estrangeiros por Público (2014–2026)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Público total (milhões de espectadores)')

plt.tight_layout()
salvar_figura('07_top10_filmes_estrangeiros.png')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.4 — DOMINÂNCIA AMERICANA NO SEGMENTO ESTRANGEIRO
# ═══════════════════════════════════════════════════════════════
# Dentro dos filmes estrangeiros, os EUA dominam de forma
# absolutamente esmagadora. Esta célula quantifica isso.

pub_por_pais = (
    df[df['ORIGEM'] == 'Estrangeiro']
      .groupby('PAIS_OBRA', observed=True)['PUBLICO']
      .sum()
      .sort_values(ascending=False)
)

total_estrangeiro = pub_por_pais.sum()
top5_paises = pub_por_pais.head(5)

print('Participação por país no público estrangeiro:')
for pais, pub in top5_paises.items():
    print(f'  {pais:<30s} {pub/1_000_000:>7.1f}M  ({pub/total_estrangeiro*100:.1f}%)')
outros = pub_por_pais.iloc[5:].sum()
print(f'  {"Demais países":<30s} {outros/1_000_000:>7.1f}M  ({outros/total_estrangeiro*100:.1f}%)')

# Salvar top filmes consolidado
top_todos = pd.concat([
    top_nacionais.assign(origem='Nacional'),
    top_estrangeiros.assign(origem='Estrangeiro'),
])
top_todos.to_csv(OUTPUT_DATA_DIR / 'top_filmes.csv', index=False)
print('\nTop filmes salvo em outputs/processados/top_filmes.csv')